# Machine Learning and Explainable AI for Fraud Detection

**Module:** RPDA8412 Research Project  
**Student:** Siyabonga Masethabe Londani Mfusi  
**Student number:** ST10295312

This notebook reproduces the report's leakage-safe comparison of five classifiers and the SHAP/LIME analysis of the selected Random Forest model. The public Kaggle dataset is a methodological benchmark; it is not treated as evidence of current South African fraud prevalence.


## Research questions

1. How do Logistic Regression, Decision Tree, Random Forest, linear SVM and k-NN compare across precision, recall, F1-score, ROC-AUC and PR-AUC?
2. Which features drive the strongest model's global and individual predictions according to SHAP and LIME, and how consistent are the explanations?
3. What do the performance and explanation findings imply for fraud-analytics practice and POPIA-informed governance in South Africa?


In [ ]:
from pathlib import Path
from itertools import combinations
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "creditcard.csv"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")


## Load and validate the data

Download the Kaggle Credit Card Fraud Detection CSV and place it at `data/creditcard.csv`. The notebook records the shape, missingness and class distribution before modelling.


In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download it from "
        "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud"
    )

df = pd.read_csv(DATA_PATH)
display(df.head())
print("Shape:", df.shape)
print("Missing cells:", int(df.isna().sum().sum()))
print("Class counts:
", df["Class"].value_counts().sort_index())


## Leakage-safe preparation

The target is separated before a stratified 80:20 split. `RobustScaler` is fitted only on the training partition for `Time` and `Amount`. Every training fraud case is retained, a reproducible sample of 30,000 legitimate training cases is used, and a manual SMOTE implementation balances the training set. The untouched test set remains at natural prevalence.


In [ ]:
def manual_smote(x_min, target_count, k=5, seed=SEED):
    rng = np.random.default_rng(seed)
    nn = NearestNeighbors(n_neighbors=k + 1, n_jobs=-1).fit(x_min)
    neighbours = nn.kneighbors(x_min, return_distance=False)[:, 1:]
    needed = target_count - len(x_min)
    base = rng.integers(0, len(x_min), size=needed)
    chosen = neighbours[base, rng.integers(0, k, size=needed)]
    gaps = rng.random((needed, 1))
    synthetic = x_min[base] + gaps * (x_min[chosen] - x_min[base])
    return np.vstack([x_min, synthetic])

feature_names = [c for c in df.columns if c != "Class"]
X = df[feature_names].to_numpy(dtype=np.float32)
y = df["Class"].to_numpy(dtype=np.int8)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

scale_idx = [feature_names.index("Time"), feature_names.index("Amount")]
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[:, scale_idx] = scaler.fit_transform(X_train[:, scale_idx]).astype(np.float32)
X_test_scaled[:, scale_idx] = scaler.transform(X_test[:, scale_idx]).astype(np.float32)

rng = np.random.default_rng(SEED)
fraud_train = X_train_scaled[y_train == 1]
legit_train = X_train_scaled[y_train == 0]
legit_sample = legit_train[rng.choice(len(legit_train), size=30_000, replace=False)]
fraud_smote = manual_smote(fraud_train, 30_000)

X_balanced = np.vstack([legit_sample, fraud_smote])
y_balanced = np.concatenate([np.zeros(30_000, dtype=np.int8), np.ones(30_000, dtype=np.int8)])
order = rng.permutation(len(y_balanced))
X_balanced, y_balanced = X_balanced[order], y_balanced[order]

print("Training shape:", X_train.shape)
print("Untouched test shape:", X_test.shape)
print("Balanced training shape:", X_balanced.shape)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(x=["Legitimate", "Fraud"], y=np.bincount(y), ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("Original class distribution")
sns.barplot(x=["Legitimate", "Fraud"], y=np.bincount(y_balanced), ax=axes[1])
axes[1].set_title("Balanced training distribution")
plt.tight_layout()
plt.show()


## Five-model comparison

All models use the same balanced training data and untouched test records. F1-score and PR-AUC are the primary selection measures because the positive class is extremely rare; confusion-matrix counts make the operational error trade-off visible.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, solver="lbfgs", random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=12, min_samples_leaf=5, random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=15, min_samples_leaf=2,
        random_state=SEED, n_jobs=-1
    ),
    "Linear SVM": LinearSVC(C=1.0, random_state=SEED, max_iter=5000),
    "k-NN": KNeighborsClassifier(n_neighbors=5, weights="distance", n_jobs=1),
}

rows, confusion = [], {}
fitted_models = {}
for name, model in models.items():
    model.fit(X_balanced, y_balanced)
    fitted_models[name] = model
    pred = model.predict(X_test_scaled)
    score = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else model.decision_function(X_test_scaled)
    cm = confusion_matrix(y_test, pred)
    confusion[name] = cm
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, score),
        "PR-AUC": average_precision_score(y_test, score),
        "TN": int(cm[0, 0]), "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]), "TP": int(cm[1, 1]),
    })

metrics = pd.DataFrame(rows).set_index("Model").sort_values("F1", ascending=False)
display(metrics.round(3))
metrics.to_csv(OUTPUT_DIR / "model_metrics.csv")


In [ ]:
metrics[["Recall", "F1", "PR-AUC"]].plot(kind="bar", figsize=(11, 5))
plt.title("Model performance on the untouched test set")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

display(metrics[["TN", "FP", "FN", "TP"]])


### Model-selection interpretation

Random Forest is selected because it achieves the strongest combined F1-score and PR-AUC while producing far fewer false positives than the other high-recall models. This is a benchmark result under the documented conditions, not a claim of universal superiority or deployment readiness.


## SHAP analysis of the selected Random Forest

The global analysis uses all 98 fraud observations in the test set plus a reproducible sample of 402 legitimate observations. Mean absolute SHAP values rank the transformed variables that most influenced model outputs in this sample.


In [ ]:
import shap

rf = fitted_models["Random Forest"]
rf_pred = rf.predict(X_test_scaled)
rf_prob = rf.predict_proba(X_test_scaled)[:, 1]

fraud_test_ids = np.flatnonzero(y_test == 1)
legit_test_ids = np.flatnonzero(y_test == 0)
global_ids = np.concatenate([fraud_test_ids, rng.choice(legit_test_ids, size=402, replace=False)])
X_explain = X_test_scaled[global_ids]

shap_explainer = shap.TreeExplainer(rf)
raw = np.asarray(shap_explainer.shap_values(X_explain, check_additivity=False))
shap_values = raw[:, :, 1] if raw.ndim == 3 else raw
importance = np.mean(np.abs(shap_values), axis=0)
order = np.argsort(importance)[::-1]
shap_table = pd.DataFrame({
    "Feature": np.asarray(feature_names)[order],
    "Mean absolute SHAP": importance[order],
})
display(shap_table.head(10))

shap_table.head(12).sort_values("Mean absolute SHAP").plot(
    x="Feature", y="Mean absolute SHAP", kind="barh", legend=False, figsize=(8, 5)
)
plt.title("Random Forest global feature importance using SHAP")
plt.tight_layout()
plt.show()


## Local SHAP and LIME agreement

One true positive, false positive, true negative and false negative are examined. LIME is repeated five times with documented seeds. Top-five SHAP/LIME overlap is measured with Jaccard similarity, and mean pairwise Jaccard similarity across LIME repeats represents local stability.


In [ ]:
from lime.lime_tabular import LimeTabularExplainer

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if a | b else 1.0

case_sets = {
    "True positive": np.flatnonzero((y_test == 1) & (rf_pred == 1)),
    "False positive": np.flatnonzero((y_test == 0) & (rf_pred == 1)),
    "True negative": np.flatnonzero((y_test == 0) & (rf_pred == 0)),
    "False negative": np.flatnonzero((y_test == 1) & (rf_pred == 0)),
}

lime_ids = np.concatenate([
    rng.choice(np.flatnonzero(y_balanced == 0), 2_500, replace=False),
    rng.choice(np.flatnonzero(y_balanced == 1), 2_500, replace=False),
])
lime_reference = X_balanced[lime_ids]

local_rows = []
for case_name, ids in case_sets.items():
    case_id = int(ids[0])
    case = X_test_scaled[case_id]
    local_raw = np.asarray(shap_explainer.shap_values(case.reshape(1, -1), check_additivity=False))
    local_shap = local_raw[0, :, 1] if local_raw.ndim == 3 else local_raw[0]
    shap_top = [feature_names[i] for i in np.argsort(np.abs(local_shap))[::-1][:5]]

    lime_runs = []
    for repeat in range(5):
        lime_explainer = LimeTabularExplainer(
            lime_reference, feature_names=feature_names,
            class_names=["Legitimate", "Fraud"], mode="classification",
            discretize_continuous=True, random_state=SEED + repeat,
        )
        explanation = lime_explainer.explain_instance(
            case, rf.predict_proba, labels=(1,), num_features=10, num_samples=2_000
        )
        lime_runs.append([feature_names[i] for i, _ in explanation.as_map()[1][:5]])

    pairwise = [jaccard(a, b) for a, b in combinations(lime_runs, 2)]
    local_rows.append({
        "Case": case_name,
        "Fraud probability": rf_prob[case_id],
        "SHAP top five": ", ".join(shap_top),
        "LIME top five": ", ".join(lime_runs[0]),
        "SHAP-LIME agreement": jaccard(shap_top, lime_runs[0]),
        "LIME repeat stability": np.mean(pairwise),
    })

local_results = pd.DataFrame(local_rows)
display(local_results)
local_results.to_csv(OUTPUT_DIR / "xai_case_comparison.csv", index=False)


## Conclusions and limitations

- Random Forest provided the strongest balance between fraud detection and false-alarm control under the common test conditions.
- Accuracy concealed important minority-class differences; PR-AUC, F1-score and confusion-matrix counts were more informative.
- SHAP and LIME agreed more strongly for the selected clear cases than for the selected errors, supporting explanation-quality checks and human review.
- The anonymised PCA features prevent substantive business interpretation.
- The historical European dataset cannot establish present South African model performance. Recent locally governed data, threshold validation, fairness checks, analyst review and POPIA-informed controls are required before operational use.
